In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType
from pyspark.sql import Row

catalog_name = "ecommerce"

In [0]:
gold_date_df = spark.read.table(f"{catalog_name}.silver.slv_date_clean")

display(gold_date_df.limit(5))

In [0]:
gold_date_df = gold_date_df.withColumn("trad_date_id", F.date_format(F.col("date_id"), "yyyyMMdd").cast(IntegerType()))

gold_date_df = gold_date_df.withColumn("month_name", F.date_format(F.col("date_id"), "MMMM"))

gold_date_df = gold_date_df.withColumn(
    "is_weekend",
    F.when(F.col("day_name").isin("Saturday", "Sunday"), 1).otherwise(0)
)

display(gold_date_df.limit(10))

In [0]:
gold_date_column_order = [
    "date_id",
    "trad_date_id",
    "year",
    "quarter",
    "month_name",
    "week_of_year",
    "day_name",
    "is_weekend",
    "_ingested_at",
    "_source_file"
]

gold_date_df = gold_date_df.select(gold_date_column_order)

display(gold_date_df.limit(10))

In [0]:
gold_date_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_date")